# ☁️ VACA QLoRA Round-1 Trainer (GPU Cloud)

Fine-tunes **Qwen2.5-7B-Instruct-Uncensored** (4-bit QLoRA) on the VACA campaign-50 dataset — round 1, LR 2e-4, 2 epochs, seq len 2048, LoRA r8/α16, GGUF q4_k_m export.

**Works on:** Google Colab (T4 free), Kaggle, Lightning AI, RunPod, Vast.ai, Modal.

**You need:** the `campaign50-dataset.zip` (train/val/test jsonl) in your browser downloads. Upload it in Step 2.

**Run time:** ~4–6 h on a T4/P100, ~1–2 h on an A100/L40S.

---

**Steps:** ▶ Run Step 1 (install) → ▶ Run Step 2 (upload dataset zip) → ▶ Run Step 3 (train) → ▶ Run Step 4 (download `results.zip`).

**Output:** `results.zip` contains `adapter/` (~80 MB) + the tuned `Qwen2.5-7B-Instruct-Uncensored-Q4_K_M.gguf` (~4.4 GB).

## Step 1 — Install dependencies
Run this once. ~2–3 min. Colab free (T4, 16 GB) is enough for this config.

In [ ]:
!pip install -q unsloth trl transformers datasets accelerate
import torch
print('CUDA:', torch.cuda.is_available(), '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'No GPU detected — change runtime type to GPU!'

## Step 2 — Upload the dataset zip
Click the folder icon → upload → pick `campaign50-dataset.zip`, **then** run this cell. It extracts `train.jsonl`, `val.jsonl`, `test.jsonl` into `./dataset/`.

In [ ]:
from google.colab import files
import zipfile, pathlib
uploaded = files.upload()          # pick campaign50-dataset.zip
zip_path = next(iter(uploaded))
pathlib.Path('dataset').mkdir(exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall('dataset')
print(sorted(p.name for p in pathlib.Path('dataset').glob('*.jsonl')))

## Step 3 — Train (round 1)
**Step 3a** writes the trainer script to disk. **Step 3b** runs it. Defaults match the local run: `--rounds 1 --lr 2e-4 --epochs 2 --max-seq-length 2048`. To run more rounds, change `--rounds` (e.g. `--rounds 3` → 2e-4 → 1e-4 → 5e-5).

In [ ]:
%%writefile train_round1.py
#!/usr/bin/env python3
"""
=============================================================================
  ☁️  Cloud QLoRA Round-1 Trainer — VACA / Veronica
  ==================================================
  Self-contained fine-tune of Qwen2.5-7B-Instruct-Uncensored (4-bit QLoRA),
  matching the LOCAL round-1 run exactly:
      LR 2e-4 · 2 epochs · max_seq_length 2048 · LoRA r8/α16/d0.05
      batch 1 · grad-accum 8 · adamw_8bit · GGUF q4_k_m export

  Designed to be uploaded to free / rented GPU services:
      Google Colab  ·  Kaggle  ·  Lightning AI  ·  RunPod  ·  Vast.ai  ·  Modal

  Requirements (single GPU, 16 GB VRAM is plenty):
      Python 3.10+ · CUDA GPU (T4/P100/A100/L4/3060) · ~15 GB free disk

  Usage:
      python train_round1.py --dataset train.jsonl
      python train_round1.py --dataset train.jsonl --out-dir ./out --rounds 1
      python train_round1.py --dry-run          # print config, no GPU work

  Outputs (in --out-dir):
      adapter/                      LoRA adapter + tokenizer (~80 MB)
      Qwen2.5-7B-Instruct-Uncensored-Q4_K_M.gguf   ← deployable tuned model
      results.zip                   everything zipped for download
=============================================================================
"""

import argparse
import gc
import json
import os
import shutil
import sys
import time
import zipfile
from datetime import datetime
from pathlib import Path

SYSTEM_PROMPT = (
    "You are a Visual AI Architect assistant specializing in GUI development. "
    "You build complete web applications, explain UI patterns, generate code, "
    "and help with frontend development."
)

# LR halving ladder matches the local engine (2e-4 → 1e-4 → 5e-5 …).
# Round 1 always uses --lr (default 2e-4).


# ─── Dependencies ───────────────────────────────────────────────────────────

def ensure_deps():
    """Import heavy libs with a helpful install hint if missing."""
    try:
        import torch, datasets, transformers, trl  # noqa
        from unsloth import FastLanguageModel, is_bfloat16_supported  # noqa
    except ImportError as e:
        print("❌ Missing dependency. In a notebook run first:")
        print('   !pip install -q unsloth trl transformers datasets accelerate')
        print(f"   (original error: {e})")
        sys.exit(1)


# ─── Dataset ────────────────────────────────────────────────────────────────

def load_dataset(path: str, tokenizer, max_seq_length: int):
    """Load instruction/input/output JSONL, tokenize with assistant-only
    label masking (prompt-length detection, same as the local engine)."""
    p = Path(path)
    if not p.exists():
        print(f"❌ Dataset not found: {path}")
        sys.exit(1)

    records = []
    with open(p, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            instruction = rec.get("instruction", "")
            input_text = rec.get("input", "")
            output_text = rec.get("output", "")

            if not output_text or len(output_text) < 10:
                continue

            user_text = instruction
            if input_text and input_text.strip() and input_text != "Auto-learned conversation":
                user_text = f"{instruction}\n\n{input_text}"

            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_text.strip()},
                {"role": "assistant", "content": output_text.strip()},
            ]

            prompt_ids = tokenizer.apply_chat_template(
                messages[:-1], tokenize=True, add_generation_prompt=True
            )
            prompt_len = len(prompt_ids)

            full_ids = tokenizer.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=False
            )
            if len(full_ids) > max_seq_length:
                full_ids = full_ids[:max_seq_length]

            labels = [-100] * min(prompt_len, len(full_ids)) + \
                     full_ids[min(prompt_len, len(full_ids)):]

            records.append({
                "input_ids": full_ids,
                "attention_mask": [1] * len(full_ids),
                "labels": labels,
            })

    if not records:
        print("❌ No valid records found in dataset!")
        sys.exit(1)

    print(f"  Loaded {len(records)} records from {p.name}")

    from datasets import Dataset as HFDataset
    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id or 0
    for rec in records:
        rec["input_ids"] = (rec["input_ids"] + [pad_id] * max_seq_length)[:max_seq_length]
        rec["attention_mask"] = (rec["attention_mask"] + [0] * max_seq_length)[:max_seq_length]
        rec["labels"] = (rec["labels"] + [-100] * max_seq_length)[:max_seq_length]

    ds = HFDataset.from_list(records)
    return ds.train_test_split(test_size=0.1, seed=42)


# ─── Evaluation ─────────────────────────────────────────────────────────────

def evaluate(model, val_dataset, max_samples: int = 60):
    import torch
    model.eval()
    device = next(model.parameters()).device
    total, correct, examples, correct_ex = 0, 0, 0, 0
    with torch.no_grad():
        for i in range(min(len(val_dataset), max_samples)):
            rec = val_dataset[i]
            input_ids = torch.tensor(rec["input_ids"], device=device).unsqueeze(0)
            attention_mask = torch.tensor(rec["attention_mask"], device=device).unsqueeze(0)
            labels = torch.tensor(rec["labels"], device=device).unsqueeze(0)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits[..., :-1, :], dim=-1)
            labels = labels[..., 1:]
            mask = labels != -100
            tc = ((preds == labels) & mask).sum().item()
            tt = mask.sum().item()
            correct += tc
            total += tt
            if ((preds == labels) & mask).all() and mask.any():
                correct_ex += 1
            examples += 1
    model.train()
    acc = round(correct / total, 4) if total else 0
    em = round(correct_ex / max(examples, 1), 4)
    return acc, em


# ─── Main ───────────────────────────────────────────────────────────────────

def main():
    ap = argparse.ArgumentParser(description="Cloud QLoRA round-1 trainer (VACA)")
    ap.add_argument("--dataset", default="train.jsonl",
                    help="Path to instruction/input/output JSONL dataset")
    ap.add_argument("--out-dir", default="out", help="Output directory")
    ap.add_argument("--rounds", type=int, default=1, help="Number of LR-decay rounds (default 1)")
    ap.add_argument("--epochs", type=int, default=2, help="Epochs per round")
    ap.add_argument("--lr", type=float, default=2e-4, help="Round-1 learning rate")
    ap.add_argument("--max-seq-length", type=int, default=2048, help="Sequence length (2048, NOT 4096)")
    ap.add_argument("--batch-size", type=int, default=1, help="Per-device batch size")
    ap.add_argument("--grad-accum", type=int, default=8, help="Gradient accumulation steps")
    ap.add_argument("--dry-run", action="store_true", help="Print config and exit (no GPU)")
    args = ap.parse_args()

    print("=" * 60)
    print("  ☁️  Cloud QLoRA Round-1 Trainer (VACA)")
    print("=" * 60)
    print(f"  Dataset        : {args.dataset}")
    print(f"  Output dir     : {args.out_dir}")
    print(f"  Rounds         : {args.rounds}  (round 1 LR = {args.lr:.1e})")
    print(f"  Epochs/round   : {args.epochs}")
    print(f"  Max seq length : {args.max_seq_length}")
    print(f"  Batch / accum  : {args.batch_size} / {args.grad_accum}")
    print(f"  GGUF export    : q4_k_m")
    print("=" * 60)

    if args.dry_run:
        print("\n✅ Dry run — config OK, no GPU work performed.")
        return

    ensure_deps()

    import torch
    from unsloth import FastLanguageModel, is_bfloat16_supported
    from transformers import TrainingArguments
    from trl import SFTTrainer

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    model_name = "Orion-zhen/Qwen2.5-7B-Instruct-Uncensored"
    print(f"\n  Loading model: {model_name} (4-bit QLoRA)...")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=args.max_seq_length,
        dtype=None,  # auto bf16/fp16
        load_in_4bit=True,
        device_map="auto",
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=8,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
        max_seq_length=args.max_seq_length,
    )
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"  Trainable params: {trainable:,} ({100 * trainable / total:.2f}%)")

    print(f"\n  Loading dataset: {args.dataset}")
    split = load_dataset(args.dataset, tokenizer, args.max_seq_length)
    train_ds, val_ds = split["train"], split["test"]
    print(f"  Train: {len(train_ds)} | Val: {len(val_ds)}")

    use_bf16 = is_bfloat16_supported()
    effective_batch = args.batch_size * args.grad_accum
    warmup = max(1, int(0.03 * len(train_ds) * args.epochs // effective_batch))

    base_args = dict(
        output_dir=str(out_dir),
        num_train_epochs=args.epochs,
        per_device_train_batch_size=args.batch_size,
        gradient_accumulation_steps=args.grad_accum,
        warmup_steps=warmup,
        lr_scheduler_type="constant",
        optim="adamw_8bit",
        max_grad_norm=0.3,
        weight_decay=0.01,
        logging_steps=10,
        save_steps=0,
        bf16=use_bf16,
        fp16=not use_bf16,
        dataloader_num_workers=4,
        remove_unused_columns=False,
        report_to="none",
    )

    steps_per_round = max(1, len(train_ds) // effective_batch * args.epochs)
    best_acc, best_em = 0.0, 0.0
    start = time.time()

    for round_num in range(1, args.rounds + 1):
        lr = args.lr * (0.5 ** (round_num - 1))  # 2e-4 → 1e-4 → 5e-5 ...
        print(f"\n{'─' * 50}")
        print(f"  Round {round_num}/{args.rounds} | LR: {lr:.1e} | steps: {steps_per_round}")
        print(f"{'─' * 50}")

        trainer = SFTTrainer(
            model=model,
            tokenizer=tokenizer,
            args=TrainingArguments(learning_rate=lr, **base_args),
            train_dataset=train_ds,
            max_seq_length=args.max_seq_length,
            dataset_text_field=None,
            packing=False,
        )
        trainer.train()

        acc, em = evaluate(model, val_ds)
        elapsed = (time.time() - start) / 60
        print(f"  📊 Round {round_num} ({elapsed:.1f}m): Acc={acc*100:.2f}% EM={em*100:.2f}%")
        if acc > best_acc:
            best_acc, best_em = acc, em

        gc.collect()
        torch.cuda.empty_cache()

    # ─── Save adapter + export GGUF ─────────────────────────────────────
    adapter_dir = out_dir / "adapter"
    print(f"\n  💾 Saving adapter to {adapter_dir} ...")
    model.save_pretrained(str(adapter_dir))
    tokenizer.save_pretrained(str(adapter_dir))

    print("\n  🔄 Exporting to GGUF (q4_k_m)...")
    model.save_pretrained_gguf(
        str(out_dir), tokenizer, quantization_method="q4_k_m"
    )

    # ─── Zip results ────────────────────────────────────────────────────
    ggufs = sorted(out_dir.glob("*.gguf"))
    if ggufs:
        g = ggufs[0]
        print(f"  📁 GGUF: {g.name} ({g.stat().st_size / 1e9:.2f} GB)")
    zip_name = out_dir / "results.zip"
    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
        if adapter_dir.exists():
            for f in adapter_dir.rglob("*"):
                if f.is_file():
                    zf.write(f, f.relative_to(out_dir))
        for g in ggufs:
            zf.write(g, g.name)
    print(f"  📦 Results zipped → {zip_name} ({zip_name.stat().st_size / 1e6:.1f} MB)")

    total_min = (time.time() - start) / 60
    print(f"\n{'=' * 50}")
    print(f"  ✅ Done in {total_min:.1f}m | Best Acc: {best_acc*100:.2f}% EM: {best_em*100:.2f}%")
    print(f"  Download: {zip_name}  (contains adapter/ + GGUF)")
    print(f"{'=' * 50}")


if __name__ == "__main__":
    main()


In [ ]:
!python train_round1.py --dataset dataset/train.jsonl --out-dir out --rounds 1

## Step 4 — Download results
Downloads `results.zip` (adapter + GGUF). On free tiers the GGUF (~4.4 GB) downloads via the browser — keep the tab open.

In [ ]:
from google.colab import files
files.download('out/results.zip')
print('✅ Downloaded. adapter/ = LoRA (~80 MB), *.gguf = tuned model (~4.4 GB)')